# Exercise 1 — Introduction to machine learning & logistic regression

This exercise accompanies **Lecture 1**. You will:

- implement **logistic regression from scratch** (sigmoid, cross-entropy, gradient descent) and compare it against scikit-learn,
- see **where linear classifiers hit their limit** on non-linear data,
- train and evaluate a classifier on a **real medical dataset** (breast cancer),
- discover why **accuracy can be misleading** on imbalanced data.

**Datasets:** synthetic Gaussian blobs, `make_moons`, breast cancer — all bundled with scikit-learn, no downloads needed.

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Wickstrom/eugloh-machine-learning-handbook/main?urlpath=lab/tree/notebooks/01/introduction_logistic_regression_exercise.ipynb)

## Setup

Run the cell below, then work through the parts in order. Add your own code
cells below each starter — there are no hidden tests; your printed output and
plots are the feedback.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

## Part A — Logistic regression by hand (vs. scikit-learn)

### A.1 — A synthetic two-class dataset

In [ ]:
X, y = make_blobs(n_samples=200, centers=[(-2.0, -2.0), (2.0, 2.0)], cluster_std=1.2, random_state=0)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X[y == 0, 0], X[y == 0, 1], color="#dc2626", alpha=0.7, label="class 0")
ax.scatter(X[y == 1, 0], X[y == 1, 1], color="#16a34a", alpha=0.7, label="class 1")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend()
ax.set_aspect("equal")
plt.show()

### A.2 — The sigmoid

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Implement it so it works for scalars **and** numpy arrays (numpy makes this automatic).

*Check yourself:* `sigmoid(0)` should be `0.5` and `sigmoid(2)` should be about `0.8808`.

In [ ]:
# YOUR CODE HERE
def sigmoid(z):
    ...

### A.3 — The cross-entropy loss

$$\mathcal{L}(w, b) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log \hat{p}_i + (1 - y_i) \log (1 - \hat{p}_i) \right]$$

with $\hat{p} = \sigma(Xw + b)$.

In [ ]:
# YOUR CODE HERE
def cross_entropy_loss(X, y, w, b):
    ...

### A.4 — Train with gradient descent

Gradients of the cross-entropy (derive them with the chain rule if you feel like it):

$$\nabla_w \mathcal{L} = \frac{1}{n} X^\top (\hat{p} - y), \qquad \nabla_b \mathcal{L} = \frac{1}{n} \sum_i (\hat{p}_i - y_i)$$

Update rule: `w ← w − lr · grad_w` and `b ← b − lr · grad_b`, repeated for `n_iter` epochs.

In [ ]:
def fit_logistic_gd(X, y, lr=0.5, n_iter=500):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    losses = []
    for _ in range(n_iter):
        # YOUR CODE HERE: compute p, the two gradients, update w and b,
        # and append the current cross-entropy loss to `losses`
        ...
    return w, b, losses


w_hand, b_hand, losses_hand = fit_logistic_gd(X, y)
print(f"learned w = {w_hand}, b = {b_hand:+.3f}")
# Expected: w ≈ [2.5, 3.0], b ≈ −0.8, training accuracy ≈ 0.99.

### A.5 — Did it learn?

Plot the loss curve. Then answer:

- Does the loss decrease smoothly?
- What happens if you retrain with `lr=50`? (Try it — this is why the learning rate matters.)

In [ ]:
# YOUR CODE HERE: plot losses_hand against the epoch number
...

### A.6 — Compare against scikit-learn

Fit `LogisticRegression` on the same data and compare weights, bias and accuracy
with your hand-written model.

> **Note:** scikit-learn applies L2 regularisation by default (`C = 1`), so its
> weights will be slightly *smaller* than your unregularised ones. That is expected.

In [ ]:
# YOUR CODE HERE: fit sklearn's LogisticRegression, print coef_, intercept_ and
# training accuracy next to your hand-written values
...

### A.7 — Decision boundaries side by side

Fill in the two missing lines below, then confirm that your boundary and
scikit-learn's coincide.

In [ ]:
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
    np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300),
)
grid = np.c_[xx.ravel(), yy.ravel()]

# YOUR CODE HERE (2 lines):
# p_hand    = probabilities from your hand-written model, reshaped to xx.shape
# p_sklearn = probabilities from sklearn's model, reshaped to xx.shape

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, p, title in zip(axes, [p_hand, p_sklearn], ["by hand (gradient descent)", "scikit-learn"]):
    ax.contourf(xx, yy, p, levels=20, cmap="RdYlGn", alpha=0.7)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], color="#dc2626", s=14)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], color="#16a34a", s=14)
    ax.set_title(title)
    ax.set_aspect("equal")
plt.show()

## Part B — Where linear classifiers hit their limit

### B.1 — Two moons

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.15, random_state=0)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_moons[y_moons == 0, 0], X_moons[y_moons == 0, 1], color="#dc2626", s=18, label="class 0")
ax.scatter(X_moons[y_moons == 1, 0], X_moons[y_moons == 1, 1], color="#16a34a", s=18, label="class 1")
ax.legend()
ax.set_aspect("equal")
plt.show()

### B.2 — Logistic regression on moons

Split the data (70/30, stratified), fit a `LogisticRegression`, and plot its
decision boundary with the helper below. The best straight line only reaches
about 85–88 % — the moons are simply not linearly separable.

In [ ]:
def plot_decision_boundary(model, X_data, y_data, ax, title):
    x_min, x_max = X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5
    y_min, y_max = X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="RdYlGn")
    ax.scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap="RdYlGn", s=15)
    ax.set_title(title)


# YOUR CODE HERE: split, fit, print test accuracy, plot the boundary
...

### B.3 — (Bonus) A polynomial rescue

`PolynomialFeatures(degree=...)` adds products of the original features
($x_1^2$, $x_2^2$, $x_1 x_2$, ...). A *linear* model on those expanded features
can draw *curved* boundaries. Try degrees 1, 3 and 7 — how do train vs test
accuracy behave? (This is overfitting — a sneak preview of Lecture 2.)

In [ ]:
# YOUR CODE HERE
for degree in [1, 3, 7]:
    pipe = make_pipeline(PolynomialFeatures(degree=degree), StandardScaler(), LogisticRegression(max_iter=5000))
    ...

## Part C — A real dataset: breast cancer

569 tumors, 30 numeric features, target: malignant (0) vs benign (1).

### C.1 — Load and split

In [ ]:
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

# YOUR CODE HERE: split 70/30 with stratify=y_c and random_state=0
X_tr, X_te, y_tr, y_te = ..., ..., ..., ...

print(f"train: {X_tr.shape}, test: {X_te.shape}")
# Expected: (398, 30) and (171, 30)

### C.2 — A properly pipelined classifier

The 30 features live on very different scales. Use a `Pipeline` with a
`StandardScaler` so the scaler is fitted on **training data only** (no data
leakage!), then fit and report the test accuracy.

In [ ]:
# YOUR CODE HERE
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
...
# Expected test accuracy ≈ 0.97–0.99

### C.3 — The confusion matrix

Not all errors are equal. Print the confusion matrix and the full
`classification_report`. **Question:** the classes are `malignant` (0) and
`benign` (1) — which error type is worse here, a false positive or a false
negative? Why?

In [ ]:
# YOUR CODE HERE: confusion matrix (ConfusionMatrixDisplay is nice) + classification_report
...

## Part D — The accuracy trap (imbalanced data)

### D.1 — A 95/5 world

In [ ]:
X_imb, y_imb = make_blobs(n_samples=[950, 50], centers=[(-2.0, -2.0), (2.0, 2.0)], cluster_std=1.5, random_state=0)
print(f"class counts: {np.bincount(y_imb)}")

### D.2 — Fit and be suspicious

Fit a logistic regression and look at the accuracy. Looks great, right?
Now fit a `DummyClassifier(strategy="most_frequent")` — a classifier that
**always predicts class 0** — and compare accuracies.

In [ ]:
# YOUR CODE HERE: fit both models on X_imb, y_imb and compare accuracies
...

### D.3 — Metrics that don't lie

Print the `classification_report` for the logistic regression. Focus on
class 1 (the rare class): what are its **precision** and **recall**? Would
you ship this model to detect a disease that affects 5 % of patients?

In [ ]:
# YOUR CODE HERE
...

---

Well done — you implemented logistic regression by hand and benchmarked it
against scikit-learn. Next up: **decision trees & random forests**
(`notebooks/02/decision_trees_random_forests_exercise.ipynb`).